In [3]:
# from huggingface_hub import snapshot_download

# snapshot_download(repo_id="amaai-lab/DisfluencySpeech", repo_type="dataset", local_dir="./DisfluencySpeech")

In [4]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [5]:
files = glob('DisfluencySpeech/data/*.parquet')
files

['DisfluencySpeech/data/train-00000-of-00003.parquet',
 'DisfluencySpeech/data/test-00000-of-00001.parquet',
 'DisfluencySpeech/data/train-00001-of-00003.parquet',
 'DisfluencySpeech/data/train-00002-of-00003.parquet',
 'DisfluencySpeech/data/validation-00000-of-00001.parquet']

In [6]:
df = pd.read_parquet(files[0])
df

,audio,transcript_annotated,transcript_a,transcript_b,transcript_c
0,{'bytes': b'RIFF\xec[\x04\x00WAVEfmt \x10\x00\...,"Yeah, I do <laughter>. Yes, {F uh, } I don't w...","Yeah, I do. Yes, uh, I don't work, though, but...","Yeah, I do. Yes, I don't work, though, but I u...","Yeah, I do. Yes, I don't work, though, but I u..."
1,{'bytes': b'RIFF\xa8n\x03\x00WAVEfmt \x10\x00\...,I think that's an interesting policy your comp...,I think that's an interesting policy your comp...,I think that's an interesting policy your comp...,I think that's an interesting policy your comp...
2,{'bytes': b'RIFF\xa8\xfa\x03\x00WAVEfmt \x10\x...,"Exactly. [ I, + I ] see more men, {F uh, } {D ...","Exactly. I, I see more men, uh, like participa...","Exactly. I, I see more men, participating in t...","Exactly. I see more men, participating in the ..."
3,{'bytes': b'RIFFx\x1a\x05\x00WAVEfmt \x10\x00\...,"{D Like, } {D you know, } helping to take care...","Like, you know, helping to take care of them m...","helping to take care of them more. And and, do...","helping to take care of them more. and, doing ..."
4,{'bytes': b'RIFF\xb0\xe1\x05\x00WAVEfmt \x10\x...,I'm <laughter> not that big on politics. I'm n...,I'm not that big on politics. I'm not that edu...,I'm not that big on politics. I'm not that edu...,I'm not that big on politics. I'm not that edu...
...,...,...,...,...,...
1495,{'bytes': b'RIFF<\n\x06\x00WAVEfmt \x10\x00\x0...,"Right, I've seen hail, {D you know, } {C but }...","Right, I've seen hail, you know, but usually t...","Right, I've seen hail, but usually the size, o...","Right, I've seen hail, but usually the size, o..."
1496,{'bytes': b'RIFF\x98\xdd\x03\x00WAVEfmt \x10\x...,"{C So } [ it had come, + it's a drop of water ...","So it had come, it's a drop of water that had ...","So it had come, it's a drop of water that had ...",So it's a drop of water that had come through ...
1497,{'bytes': b'RIFF\x84T\x05\x00WAVEfmt \x10\x00\...,"I guess, {C and then, } I think, {D you know, ...","I guess, and then, I think, you know, all that...","I guess, and then, I think, all that falling w...","I guess, and then, I think, all that falling w..."
1498,{'bytes': b'RIFFh\x8a\x04\x00WAVEfmt \x10\x00\...,There was no problem with it. {C But } I guess...,There was no problem with it. But I guess you ...,There was no problem with it. But I guess you ...,There was no problem with it. But I guess you ...


In [7]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['transcript_a'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"DisfluencySpeech"
            })
        
    return data

In [8]:
data = loop((files[:1], 0))

100%|██████████| 1/1 [01:20<00:00, 80.90s/it]


In [11]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 1/1 [01:23<00:00, 83.15s/it]


In [12]:
len(data)

5000

In [13]:
data[0]

{'audio_filename': 'DisfluencySpeech_audio/DisfluencySpeech-data-train-00000-of-00003_0.mp3',
 'text': "Yeah, I do. Yes, uh, I don't work, though, but I used to work and, when I had two children.",
 'speaker': 'DisfluencySpeech'}

In [14]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'DisfluencySpeech_audio/DisfluencySpeech-data-train-00000-of-00003_0.mp3',
 'text': "Yeah, I do. Yes, uh, I don't work, though, but I used to work and, when I had two children.",
 'speaker': 'DisfluencySpeech'}

In [15]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'DisfluencySpeech')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 303.41ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  411kB /  411kB, 1.43MB/s  
Processing Files (1 / 1): 100%|██████████|  411kB /  411kB, 1.03MB/s  
New Data Upload: 100%|██████████|  411kB /  411kB, 1.03MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.23 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/e5ba4d4862cb4a2dec7b8d5dc3bbbd6f17c13689', commit_message='Upload dataset', commit_description='', oid='e5ba4d4862cb4a2dec7b8d5dc3bbbd6f17c13689', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [16]:
audio_files = [d['audio_filename'] for d in data]

with open('DisfluencySpeech-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [18]:
folders = glob('DisfluencySpeech_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

DisfluencySpeech_audio_neucodec
DisfluencySpeech_audio


In [19]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('DisfluencySpeech_audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  94%|█████████▍| 6.24MB / 6.63MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 6.63MB / 6.63MB, 1.97MB/s  
Processing Files (1 / 1): 100%|██████████| 6.63MB / 6.63MB, 1.97MB/s  
New Data Upload: 100%|██████████| 6.63MB / 6.63MB, 1.97MB/s  
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  19%|█▉        | 48.8MB /  254MB,   ???B/s  
Processing Files (0 / 1):  64%|██████▍   |  164MB /  254MB,  573MB/s  
Processing Files (0 / 1): 100%|█████████▉|  253MB /  254MB,  509MB/s  
Processing Files (0 / 1): 100%|█████████▉|  253MB /  254MB,  255MB/s  
Processing Files (1 / 1): 100%|██████████|  254MB /  254MB,  205MB/s  
Processing Files (1 / 1): 100%|██████████|  254MB /  254MB,  171MB/s  
New Data Upload: 100%|██████████|  254MB /  254MB,  171MB/s  
